# PUBG API Data Extraction & Structuring (Python Version)
This notebook tests the extraction of PUBG match data and structures it into flattened Pandas DataFrames. It includes high-resolution telemetry data.

In [ ]:
import requests
import pandas as pd
import json
import time
from config import api_key
from IPython.display import display

In [ ]:
valid_shards = ["steam", "console"]

### Steam Data Extraction

In [ ]:
##Fetches a list of recent match IDs from the PUBG API (It is the only way to get a random, high-volume list of matches without knowing specific players beforehand)

def get_pubg_samples_steam(api_key, shard=valid_shards[0]):
    url = f"https://api.pubg.com/shards/{shard}/samples"
    headers = {
        "Authorization": f"Bearer {api_key}", 
        "Accept": "application/vnd.api+json"
    }
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error: {response.status_code} - {response.text}")
        return None

In [ ]:
match_samples = get_pubg_samples_steam(api_key)

print(match_samples)


In [ ]:
matches_list = match_samples['data']['relationships']['matches']['data']
match_samples_df = pd.DataFrame(matches_list)
match_samples_df.rename(columns={'id': 'match_id'}, inplace=True)
print(f"Total matches retrieved: {len(match_samples_df)}")
display(match_samples_df.head())

In [ ]:
match_id = match_samples_df['match_id'].tolist()

In [ ]:
def get_match_details(api_key, match_id, shard=valid_shards[0]):
    """Fetch detailed information for a specific match."""
    url = f"https://api.pubg.com/shards/{shard}/matches/{match_id}"
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Accept": "application/vnd.api+json"
    }
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error: {response.status_code} - {response.text}")
        return None

In [ ]:
match_details = get_match_details(api_key,match_id[0])
print(match_details)

In [ ]:
match_details = get_match_details(api_key,match_id[0])
matches_details_list = match_details['data']['relationships']['assets']['data']
print(matches_details_list)

In [ ]:
def extract_telemetry_url(match_details):
    included_items = match_details.get("included", [])
    
    for item in included_items:
        is_telemetry = item.get("attributes", {}).get("name") == "telemetry"
        
        if is_telemetry:
            telemetry_url = item["attributes"]["URL"]
            print("Telemtry URL sucessfully extracted!")
            return telemetry_url
            
    print("Did not find telemetry URL in match details:(")
    return None

In [ ]:
telemetry_url = extract_telemetry_url(match_details)
print(telemetry_url)